# Session 07 — Drone Human Detection | Clean Dashboard Edition (2026)
**Google Colab · YOLOv8s + ByteTrack · Ready to Run**

This revised edition fixes overlapping labels in crowded aerial video. All person bounding boxes are drawn manually rather than using `result.plot()`.

**Classroom workflow:** Runtime → Run all → choose your drone video → view result and detection chart. First run downloads model weights. If available, select a T4 GPU under Runtime → Change runtime type.

**Key concepts:** detected people in the current frame and maximum simultaneous detections are **not** the verified total number of individuals. IDs can switch, disappear or be reassigned. Real-world counts require ground-truth validation.

## Step 1 — Install the open-source detection library
This requires internet once for installation and downloading model weights. The runtime can use CPU, but processing will be slower.

In [ ]:
%pip -q install -U ultralytics

## Step 2 — Upload your original drone video
Upload the raw `.mp4`, `.mov`, or `.avi` file. This cell validates that OpenCV can read frames before any inference starts.

In [ ]:
import os, cv2, numpy as np, torch, subprocess, time
from pathlib import Path
from google.colab import files
from IPython.display import display, Video

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No video uploaded. Run this cell again and select a video file.')
dr_set = {'.mp4','.mov','.avi','.mkv','.webm'}
choices = [name for name in uploaded if Path(name).suffix.lower() in dr_set]
if not choices:
    raise ValueError('Please upload a video: MP4, MOV, AVI, MKV, or WEBM.')
drone_input = choices[0]
cap = cv2.VideoCapture(drone_input)
ok, frame = cap.read()
if not ok or frame is None:
    cap.release()
    raise RuntimeError(f'Video cannot be read: {drone_input}. Re-encode as MP4 (H.264) and retry.')
fps = cap.get(cv2.CAP_PROP_FPS)
if not np.isfinite(fps) or fps < 1 or fps > 240: fps = 24.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
nframes = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print('Uploaded:', drone_input)
print(f'Resolution: {width} x {height} | FPS: {fps:.1f} | Estimated frames: {nframes}')
print('Inference device:', 'CUDA GPU' if torch.cuda.is_available() else 'CPU (slower)')

## Step 3 — Choose visual settings and run YOLO + ByteTrack

Use the three booleans below to control **bounding boxes**, **ID labels**, and **confidence**. You can set them in Colab before running this cell. The default settings are optimized for crowded video:
- Green, thin bounding boxes
- Small, compact `ID 12` labels instead of long overlapping strings
- Confidence hidden by default
- Fixed dashboard top-left with *people visible now* and *maximum observed*

**Teaching tip:** To compare presentations, change `SHOW_IDS` or `SHOW_CONFIDENCE` and rerun Step 3. Avoid reuploading the video. The final MP4 is shown automatically.

In [ ]:
from ultralytics import YOLO

# @title YOLO + ByteTrack: model & display options
MODEL_NAME = 'yolov8s.pt'          # @param {type:"string"}
CONFIDENCE = 0.18                  # @param {type:"number"}
IMAGE_SIZE = 960                   # @param {type:"integer"}
MAX_FRAMES = 300                   # @param {type:"integer"}
SHOW_BOXES = True                  # @param {type:"boolean"}
SHOW_IDS = True                    # @param {type:"boolean"}
SHOW_CONFIDENCE = False            # @param {type:"boolean"}
SHOW_DASHBOARD = True              # @param {type:"boolean"}
BOX_THICKNESS = 2                  # @param {type:"integer"}

# The model has to be downloaded once. No HOG fallback.
model = YOLO(MODEL_NAME)
source = cv2.VideoCapture(drone_input)
if not source.isOpened():
    raise RuntimeError('Cannot reopen uploaded drone video. Run Step 2 again.')

raw_output = 'drone_yolo_clean_raw.mp4'
final_output = 'drone_yolo_clean_result.mp4'
writer = cv2.VideoWriter(raw_output, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
if not writer.isOpened():
    source.release()
    raise RuntimeError('Cannot initialize output writer.')

def draw_clean_annotations(frame, boxes, id_values, scores, frame_count, max_count):
    """Draw compact detections and a fixed dashboard without result.plot() labels."""
    canvas = frame.copy()
    thickness = max(1, int(BOX_THICKNESS))
    for index, (x1, y1, x2, y2) in enumerate(boxes):
        x1, x2 = sorted((max(0, min(int(x1), width-1)), max(0, min(int(x2), width-1))))
        y1, y2 = sorted((max(0, min(int(y1), height-1)), max(0, min(int(y2), height-1))))
        if SHOW_BOXES:
            cv2.rectangle(canvas,(x1,y1),(x2,y2),(50,220,110),thickness)
        parts=[]
        if SHOW_IDS and id_values[index] is not None:
            parts.append(f'ID {id_values[index]}')
        if SHOW_CONFIDENCE:
            parts.append(f'{scores[index]:.2f}')
        if parts:
            label=' '.join(parts)
            scale=max(0.38, min(0.52, width/1800))
            (tw,th),baseline=cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,scale,1)
            # Put label above its box when room exists, otherwise below top edge.
            tx=max(2,min(x1,width-tw-6))
            ty=y1-6 if y1 > th+10 else min(height-th-6, y1+th+12)
            cv2.rectangle(canvas,(tx-2,ty-th-4),(tx+tw+3,ty+baseline+2),(19,65,35),-1)
            cv2.putText(canvas,label,(tx,ty),cv2.FONT_HERSHEY_SIMPLEX,scale,(245,255,245),1,cv2.LINE_AA)
    if SHOW_DASHBOARD:
        panel_w=min(width-16,440)
        panel_h=min(height-12,105)
        shade=canvas.copy()
        cv2.rectangle(shade,(8,8),(8+panel_w,8+panel_h),(13,29,27),-1)
        cv2.addWeighted(shade,.82,canvas,.18,0,canvas)
        font_scale=min(0.72,max(.40,width/1250))
        lines=[('DRONE HUMAN MONITORING',(255,255,255)),
               (f'People detected now: {frame_count}',(105,245,161)),
               (f'Maximum detected: {max_count}',(105,245,161))]
        step=max(18,int(panel_h/3))
        for i,(line,color) in enumerate(lines):
            cv2.putText(canvas,line,(18,20+step*i),cv2.FONT_HERSHEY_SIMPLEX,font_scale,color,2 if i==0 else 1,cv2.LINE_AA)
    return canvas

counts=[]
seen_ids=set()
processed=0
maximum_detected=0
last_status=time.monotonic()
try:
    while MAX_FRAMES is None or MAX_FRAMES <= 0 or processed < MAX_FRAMES:
        ok, image=source.read()
        if not ok:
            break
        prediction=model.track(
            image,persist=True,classes=[0],conf=CONFIDENCE,imgsz=IMAGE_SIZE,
            tracker='bytetrack.yaml',verbose=False,
            device=0 if torch.cuda.is_available() else 'cpu'
        )[0]
        boxes=[]; scores=[]; track_ids=[]
        if prediction.boxes is not None and len(prediction.boxes):
            boxes=prediction.boxes.xyxy.cpu().numpy().tolist()
            scores=prediction.boxes.conf.cpu().numpy().tolist()
            if prediction.boxes.id is not None:
                track_ids=[int(v) for v in prediction.boxes.id.cpu().tolist()]
            else:
                track_ids=[None]*len(boxes)
        seen_ids.update(v for v in track_ids if v is not None)
        visible=len(boxes)
        counts.append(visible)
        maximum_detected=max(maximum_detected,visible)
        canvas=draw_clean_annotations(image,boxes,track_ids,scores,visible,maximum_detected)
        writer.write(canvas)
        processed+=1
        if time.monotonic()-last_status>15:
            print(f'Processed {processed} frames ...')
            last_status=time.monotonic()
finally:
    source.release();writer.release()

if processed==0:
    raise RuntimeError('No frames processed. Check uploaded video and rerun Step 2.')

subprocess.run([
    'ffmpeg','-y','-loglevel','error','-i',raw_output,
    '-c:v','libx264','-pix_fmt','yuv420p','-movflags','+faststart',final_output
],check=True)
print(f'Processed {processed} frames | maximum detected per frame: {maximum_detected}')
print(f'Average detections/frame: {np.mean(counts):.1f} | tracker IDs assigned: {len(seen_ids)}')
print('Tracker IDs are NOT verified unique people.')
print('Output:',final_output)
display(Video(final_output,embed=True,width=800))

## Step 4 — Analyze and download

The chart displays frame-by-frame detection counts, not verified attendance. Compare crowd density with detection dips and ask whether camera motion, blur, or occlusion might explain them. You can download the annotated MP4 after the run.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10,3.6))
plt.plot(np.arange(1,len(counts)+1),counts,linewidth=1.7)
plt.axhline(np.mean(counts),linestyle='--',linewidth=1,label=f'Average: {np.mean(counts):.1f}')
plt.title('People Detected per Frame — Drone Footage')
plt.xlabel('Processed frame')
plt.ylabel('Person detections')
plt.legend()
plt.grid(alpha=.25)
plt.tight_layout()
plt.show()
print('Maximum simultaneous detections:',max(counts))
print('Mean detections/frame:',round(float(np.mean(counts)),1))
# Uncomment to download annotated MP4:
# files.download(final_output)

## Guided discussion (5 minutes)
1. Why did HOG fail while YOLO may detect some people in this aerial scene?
2. How do altitude, image resolution, occlusion, camera motion, and detection confidence change the results?
3. Why is the number of tracking IDs **not** necessarily the number of unique humans?
4. How could we evaluate detector quality? (Annotation of ground-truth people; precision, recall, F1, and mAP.)
5. What privacy and consent safeguards would be necessary for an actual drone monitoring system?

**Instructor note:** model weights are downloaded from the internet on first use, so have Wi-Fi ready. This notebook does **not** use HOG as a fallback and therefore will not silently display zero detections from that baseline.